# 🌍 TWS Forecasting Challenge – Starter Notebook (flat CSV release)

This notebook provides a simple baseline pipeline for predicting next-month Total Water
Storage (TWS).

**Updated for the flat CSV data release.** The competition data is now distributed as flat,
one-row-per-sample CSV files (`Train.csv`, `Test.csv`) instead
of gridded NetCDF files, so this notebook no longer needs `xarray` or any grid-indexing
logic — every feature is already a column.

One more thing that's different from earlier releases: roughly two-thirds of `TWS_t` values
in the test set are intentionally masked (`NaN`). This is not missing data — it's there to
prevent `TWS_t` at one test row from revealing the hidden target of another test row at the
same location the month before. See the `TWS_t_masked` column. Your model needs to handle
these missing values.

We will:
- Explore the data
- Visualize spatial and temporal patterns
- Build a simple baseline model
- Generate a submission

The goal is to improve over a strong persistence baseline.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:
DATA_DIR = Path("../data")  # change if needed


TRAIN_FEATURES_PATH = DATA_DIR / "Train.csv"
TEST_FEATURES_PATH = DATA_DIR / "Test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "SampleSubmission.csv"

In [ ]:
## Checking the file sizes
for file_path in [TRAIN_FEATURES_PATH, TEST_FEATURES_PATH, SAMPLE_SUBMISSION_PATH]:
    file_size = file_path.stat().st_size
    file_size_mb = file_size / (1024 * 1024)
    print(f"{file_path.name}: {file_size_mb:.2f} MB")


## 🔍 Dataset Overview

We are working with flat, tabular monthly data. Each row is one `(time, lat, lon)` sample,
identified by `ID` (format `YYYYMMDD_lat_lon`).

Available columns:
- `TWS_t` — current-month Total Water Storage( **masked in ~66%
  of test rows**, see `TWS_t_masked`)
- `SPEI_01_t`, `SPEI_03_t`, `SPEI_06_t`, `SPEI_12_t` — drought/wetness indices at multiple
  timescales
- `SOIL_MOISTURE_t` — current-month soil moisture
- `month_sin`, `month_cos` — cyclical month-of-year encoding
- `target` — TWS one month ahead (train only; this is what you're predicting)
- `TWS_t_masked` — boolean flag, test only, `True` where `TWS_t` has been masked


In [ ]:
def normalise_id_column(df: pd.DataFrame) -> pd.DataFrame:
    """Some releases use 'sample_id' instead of 'ID' for the test file — normalise it."""
    if "sample_id" in df.columns and "ID" not in df.columns:
        out = df.copy()
        out = out.rename(columns={"sample_id": "ID"})
        return out
    return df


train_features = normalise_id_column(pd.read_csv(TRAIN_FEATURES_PATH, parse_dates=["time"]))
test_features = normalise_id_column(pd.read_csv(TEST_FEATURES_PATH, parse_dates=["time"]))
sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("train_features:", train_features.shape)
print("test_features: ", test_features.shape)
print("sample_submission:", sample_submission_df.shape)

print("\nTrain columns:", train_features.columns.tolist())
print("Test columns: ", test_features.columns.tolist())


In [ ]:
train_features.head()


In [ ]:
print("Test TWS_t masking:")
print(test_features["TWS_t_masked"].value_counts())
print(f"Masked share: {test_features['TWS_t_masked'].mean():.1%}")


In [ ]:
test_features.head()

In [ ]:
available_train = set(train_features.columns)
available_test = set(test_features.columns)
available_common = sorted(
    (available_train & available_test) - {"ID", "time", "lat", "lon", "TWS_t_masked"}
)

print("Available common feature columns:")
print(available_common)


## 🌍 Spatial Patterns

Let's visualize the spatial distribution of Total Water Storage (TWS) for a single month.
This helps us understand:
- Large-scale geographic structure
- Differences between regions (e.g. land vs ocean, wet vs dry zones)


In [ ]:
def pivot_grid(df: pd.DataFrame, time_value, value_col: str) -> pd.DataFrame:
    """Pivot a single time slice of the flat dataframe into a (lat x lon) grid for plotting."""
    sub = df.loc[df["time"] == time_value, ["lat", "lon", value_col]]
    return sub.pivot(index="lat", columns="lon", values=value_col).sort_index().sort_index(axis=1)


t0 = train_features["time"].min()
print("Plotting time slice:", t0)


In [ ]:
grid_tws = pivot_grid(train_features, t0, "TWS_t")

plt.figure(figsize=(10, 5))
plt.imshow(grid_tws.values, cmap="RdBu", origin="lower",
           extent=[grid_tws.columns.min(), grid_tws.columns.max(),
                   grid_tws.index.min(), grid_tws.index.max()])
plt.colorbar(label="TWS(t)")
plt.title(f"TWS(t) at time {str(t0)[:10]}")
plt.xlabel("lon")
plt.ylabel("lat")
plt.show()


## 🌧️ Climate Forcing (SPEI)

We compare TWS with SPEI (Standardized Precipitation-Evapotranspiration Index).

SPEI represents drought/wetness conditions:
- Positive → wetter than normal
- Negative → drier than normal

This gives insight into how climate affects water storage.


In [ ]:
grid_spei = pivot_grid(train_features, t0, "SPEI_01_t")

fig, axs = plt.subplots(1, 2, figsize=(14, 5))

im0 = axs[0].imshow(grid_tws.values, cmap="RdBu", origin="lower")
axs[0].set_title("TWS(t)")
plt.colorbar(im0, ax=axs[0], fraction=0.046)

im1 = axs[1].imshow(grid_spei.values, cmap="BrBG", origin="lower")
axs[1].set_title("SPEI_01(t)")
plt.colorbar(im1, ax=axs[1], fraction=0.046)

plt.show()


## 📈 Temporal Dynamics

We inspect a time series at a single location.

Key observation:
- TWS has strong temporal memory
- Changes are gradual rather than abrupt

This suggests that **past TWS is a strong predictor of future TWS**.


In [ ]:
lat0 = 10.5
lon0 = 0.5

# Grid is on fixed 1-degree centres; snap the requested point to the nearest grid cell.
nearest_lat = train_features["lat"].sub(lat0).abs().idxmin()
nearest_lat = train_features.loc[nearest_lat, "lat"]
nearest_lon = train_features["lon"].sub(lon0).abs().idxmin()
nearest_lon = train_features.loc[nearest_lon, "lon"]

ts = train_features[(train_features["lat"] == nearest_lat) & (train_features["lon"] == nearest_lon)].sort_values("time")

plt.figure(figsize=(10, 4))
plt.plot(ts["time"], ts["TWS_t"], label="TWS(t)")
plt.plot(ts["time"], ts["SPEI_01_t"], label="SPEI_01(t)")
plt.legend()
plt.title(f"Time series at lat={nearest_lat}, lon={nearest_lon}")
plt.grid(True)
plt.show()


## 🔥 Persistence Baseline

A very strong baseline is:

TWS(t+1) ≈ TWS(t)

Since `Target` in the training file already **is** TWS at `t+1` (by construction), we can
visualize this relationship directly from `TWS_t` vs `Target` — no grid-shifting needed.

Any useful model must improve over this baseline.


In [ ]:
if 'target' in train_features:
    train_features = train_features.rename(columns={'target': 'Target'})

In [ ]:
pairs = train_features[["TWS_t", "Target"]].dropna()
n_plot = min(50000, len(pairs))
sample = pairs.sample(n=n_plot, random_state=42)

plt.figure(figsize=(6, 6))
plt.scatter(sample["TWS_t"], sample["Target"], s=3, alpha=0.3)

mn, mx = sample["TWS_t"].min(), sample["TWS_t"].max()
plt.plot([mn, mx], [mn, mx], "--")

plt.xlabel("TWS(t)")
plt.ylabel("TWS(t+1) = Target")
plt.title("Persistence strength")
plt.grid(True)
plt.show()


In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "r2": r2_score(y_true, y_pred),
    }


# Test.csv hides TWS_t on ~66.5% of rows; Train.csv hides none. These two helpers
# are what keep the fit and the score honest about that gap.
TWS_MASK_RATE = 0.665


def mask_tws(df: pd.DataFrame, rate: float = TWS_MASK_RATE, seed: int = 0) -> pd.DataFrame:
    """Return a copy of `df` with TWS_t set to NaN on a random `rate` share of rows.

    Adds a TWS_t_masked flag so scores can be broken down by regime.
    """
    out = df.copy().reset_index(drop=True)
    masked = np.random.RandomState(seed).rand(len(out)) < rate
    out.loc[masked, "TWS_t"] = np.nan
    out["TWS_t_masked"] = masked
    return out


# Test.csv does NOT mask at random. It is 6 runs of consecutive months, and only the
# first month of each run keeps TWS_t; every later month is masked because its TWS_t
# *is* the previous month's hidden target. Run lengths observed in Test.csv:
TEST_RUN_LENGTHS = [1, 3, 4, 7, 1, 2]


def assign_run_layout(times, run_lengths=TEST_RUN_LENGTHS) -> pd.DataFrame:
    """Chop a list of months into runs shaped like the test set.

    Runs never straddle a calendar gap. A run's whole point is that each month
    follows the last, and the GRACE record has real holes in it — pretending two
    months a year apart are consecutive would make the chain meaningless.

    Returns one row per month with `run_id` and `gap` — months since that run's
    anchor. gap == 0 keeps TWS_t; gap >= 1 is masked, and gap is the forecast
    horizon that row really demands.
    """
    times = pd.DatetimeIndex(np.sort(np.asarray(times)))
    midx = times.year * 12 + times.month

    blocks, start = [], 0
    for i in range(1, len(times) + 1):
        if i == len(times) or midx[i] != midx[i - 1] + 1:
            blocks.append((start, i))
            start = i

    run_id = np.empty(len(times), dtype=np.int32)
    gap = np.empty(len(times), dtype=np.int32)
    r = 0
    for s, e in blocks:
        i = s
        while i < e:
            length = min(run_lengths[r % len(run_lengths)], e - i)
            run_id[i:i + length] = r
            gap[i:i + length] = np.arange(length)
            i += length
            r += 1

    return pd.DataFrame({"time": times, "run_id": run_id, "gap": gap})


def mask_tws_like_test(df: pd.DataFrame, run_lengths=TEST_RUN_LENGTHS) -> pd.DataFrame:
    """Mask TWS_t the way Test.csv does: whole months at a time.

    Per-row random masking leaves same-month neighbours visible, so imputing a masked
    cell from its neighbours would score well here. In the real test set only 0.4% of
    masked rows have any of their 4 neighbours available — that gain would be fake.
    """
    layout = assign_run_layout(df["time"].unique(), run_lengths)
    out = df.merge(layout, on="time", how="left")  # left join preserves row order
    masked = (out["gap"] > 0).to_numpy()
    out.loc[masked, "TWS_t"] = np.nan
    out["TWS_t_masked"] = masked
    return out


def augment_with_masking(df: pd.DataFrame, rate: float = TWS_MASK_RATE,
                         seed: int = 0) -> pd.DataFrame:
    """Original rows stacked with a randomly masked copy.

    HistGradientBoostingRegressor only learns a split direction for NaN if it sees
    NaN while fitting. Train.csv never does, so without this the model has no
    learned behaviour for the two-thirds of test rows where TWS_t is hidden.

    Random masking is deliberate *here*: for fitting we only need varied examples of
    "TWS_t is absent". Scoring is a different matter — see mask_tws_like_test.
    """
    return pd.concat([df, mask_tws(df, rate, seed)], ignore_index=True)


def make_model():
    # No SimpleImputer: HistGradientBoostingRegressor handles NaN natively and learns
    # which way to send missing values. Median-filling TWS_t would instead tell the
    # model that 66.5% of test rows sit at the average, which they don't.
    return Pipeline([
        ("gbr", HistGradientBoostingRegressor(
            loss="squared_error",
            learning_rate=0.05,
            max_iter=300,
            max_depth=8,
            min_samples_leaf=50,
            l2_regularization=1.0,
            random_state=42
        ))
    ])


def add_cell_id(df: pd.DataFrame) -> pd.DataFrame:
    """Attach a stable grid-cell key so runs can be chained location by location."""
    out = df.copy()
    out["cell"] = out["lat"].astype(str) + "_" + out["lon"].astype(str)
    return out


def find_runs(df: pd.DataFrame) -> list:
    """Group months into runs by reading the TWS_t masking pattern itself.

    A month whose TWS_t is mostly observed starts a run; each following
    calendar-consecutive month whose TWS_t is mostly missing joins it. Test.csv masks
    ~99.7% rather than exactly 100% of a masked month, hence the majority test.
    """
    observed = df.groupby("time")["TWS_t"].apply(lambda s: s.notna().mean() > 0.5)
    months = pd.DatetimeIndex(observed.index)
    midx = months.year * 12 + months.month

    runs, cur = [], []
    for i, m in enumerate(months):
        if observed.iloc[i] or not cur or midx[i] != midx[i - 1] + 1:
            if cur:
                runs.append(cur)
            cur = [m]
        else:
            cur.append(m)
    if cur:
        runs.append(cur)
    return runs


def predict_recursive(model, df: pd.DataFrame, feature_names: list) -> np.ndarray:
    """Forecast through each run, feeding every month's prediction forward.

    target(cell, M) IS TWS(cell, M+1), so a run is self-supplying: only the anchor
    month needs an observed TWS_t. Each step also picks up that month's real SPEI and
    soil moisture, which is what makes this beat carrying the anchor forward — the
    forcing between anchor and target is unmasked and otherwise goes unused.

    `df` needs a `cell` column (see add_cell_id).
    """
    out = pd.Series(np.nan, index=df.index, dtype=np.float64)
    tws_pos = feature_names.index("TWS_t")
    groups = df.groupby("time").groups

    for run in find_runs(df):
        carried, anchor = None, None
        for g, month in enumerate(run):
            idx = groups[month]
            block = df.loc[idx]
            # copy=True: pandas hands back a read-only view under copy-on-write, and
            # the recursive step needs to overwrite the TWS_t column.
            X = block[feature_names].to_numpy(dtype=np.float32, copy=True)

            if g == 0:
                anchor = dict(zip(block["cell"], block["TWS_t"]))
            else:
                # Previous month's prediction; fall back to the run's anchor for
                # cells the grid did not cover last month.
                filled = block["cell"].map(carried)
                filled = filled.fillna(block["cell"].map(anchor))
                X[:, tws_pos] = filled.to_numpy(dtype=np.float32)

            pred = model.predict(X)
            out.loc[idx] = pred
            carried = dict(zip(block["cell"], pred))

    return out.to_numpy()


def build_xy(df: pd.DataFrame, feature_names: list, use_latlon: bool = False):
    """Flat-file equivalent of the old NetCDF-indexing build_xy: features are already
    columns, so this is just a column selection.
    Lat/Lon should not be use as features!"""
    X = df[feature_names].to_numpy(dtype=np.float32)

    if use_latlon:
        latlon = df[["lat", "lon"]].to_numpy(dtype=np.float32)
        X = np.column_stack([X, latlon])

    # Corrected: changed "target" to "Target"
    y = df["Target"].to_numpy(dtype=np.float32) if "Target" in df.columns else None

    # persistence baseline = TWS_t
    tws_t = df["TWS_t"].to_numpy(dtype=np.float32)

    return X, y, tws_t


def plot_maps_from_predictions(df, y_true, y_pred, time_value):
    """
    Plot TWS true, TWS predicted, and error = pred - true for one time slice.

    Parameters
    ----------
    df : pandas.DataFrame
        Must contain columns: time, lat, lon
    y_true, y_pred : array-like, aligned with df's row order
    time_value : timestamp-like, one time step present in df["time"]
    """
    mask = df["time"] == time_value

    df_sub = df.loc[mask, ["time", "lat", "lon"]].copy().reset_index(drop=True)
    y_true_sub = np.asarray(y_true)[mask.values]
    y_pred_sub = np.asarray(y_pred)[mask.values]

    if len(df_sub) == 0:
        raise ValueError(f"No rows found for time={time_value}")

    lats = np.sort(df_sub["lat"].unique())
    lons = np.sort(df_sub["lon"].unique())

    lat_to_idx = {float(v): i for i, v in enumerate(lats)}
    lon_to_idx = {float(v): i for i, v in enumerate(lons)}

    H, W = len(lats), len(lons)

    true_grid = np.full((H, W), np.nan, dtype=float)
    pred_grid = np.full((H, W), np.nan, dtype=float)

    for i, row in df_sub.iterrows():
        lat_i = lat_to_idx[float(row["lat"])]
        lon_i = lon_to_idx[float(row["lon"])]

        true_grid[lat_i, lon_i] = y_true_sub[i]
        pred_grid[lat_i, lon_i] = y_pred_sub[i]

    error_grid = pred_grid - true_grid

    vmin = np.nanpercentile(y_true_sub, 1)
    vmax = np.nanpercentile(y_true_sub, 99)
    emax = np.nanpercentile(np.abs(error_grid[np.isfinite(error_grid)]), 99)

    fig, axs = plt.subplots(1, 3, figsize=(18, 5))

    im0 = axs[0].imshow(true_grid, cmap="RdBu", vmin=vmin, vmax=vmax, origin="lower")
    axs[0].set_title("TWS True")

    im1 = axs[1].imshow(pred_grid, cmap="RdBu", vmin=vmin, vmax=vmax, origin="lower")
    axs[1].set_title("TWS Predicted")

    im2 = axs[2].imshow(error_grid, cmap="coolwarm", vmin=-emax, vmax=emax, origin="lower")
    axs[2].set_title("Error (Pred - True)")

    for ax in axs:
        ax.axis("off")

    plt.colorbar(im0, ax=axs[0], fraction=0.046)
    plt.colorbar(im1, ax=axs[1], fraction=0.046)
    plt.colorbar(im2, ax=axs[2], fraction=0.046)

    plt.suptitle(f"Maps at time {pd.Timestamp(time_value).strftime('%Y-%m-%d')}")
    plt.tight_layout()
    plt.show()

In [ ]:
BASE_FEATURES = [
    "TWS_t",
    "month_sin",
    "month_cos",
]

# Note: the _tp1 (future-looking) columns from the old NetCDF release are not present in
# the flat CSV files, so they're no longer part of the optional feature set.
OPTIONAL_FEATURES = [
    "SPEI_01_t", "SPEI_03_t", "SPEI_06_t", "SPEI_12_t", "SOIL_MOISTURE_t",
]

feature_names = BASE_FEATURES + [f for f in OPTIONAL_FEATURES if f in available_common]
print(feature_names)


In [ ]:
unique_times = np.sort(train_features["time"].unique())
split_idx = int(len(unique_times) * 0.8)

fit_times = unique_times[:split_idx]
val_times = unique_times[split_idx:]

fit_df = train_features[train_features["time"].isin(fit_times)].copy().reset_index(drop=True)
val_df = train_features[train_features["time"].isin(val_times)].copy().reset_index(drop=True)

print("Fit period:", fit_df["time"].min(), "->", fit_df["time"].max())
print("Val period:", val_df["time"].min(), "->", val_df["time"].max())
print("Fit samples:", len(fit_df))
print("Val samples:", len(val_df))


In [ ]:
fit_df.head()

In [ ]:
val_df.head()

### Matching the validation set to the test set, and forecasting through it

`Train.csv` never hides `TWS_t`. `Test.csv` hides it on 66.5% of rows, **blocked by
month, not at random**. The test set is 6 runs of consecutive months; only each run's
first month keeps `TWS_t`, and every later month is masked because its `TWS_t` *is* the
previous month's hidden target:

```
2015-09 present                       (run of 1)
2016-01 present -> 02, 03 masked      (run of 3)
2016-06 present -> 07, 08, 09 masked  (run of 4)
2016-12 present -> 01..06 masked      (run of 7)
2018-07 present                       (run of 1)
2018-11 present -> 12 masked          (run of 2)
```

That identity is also the way through. Because `target(cell, M)` **is** `TWS(cell, M+1)`,
a run supplies its own missing inputs: predict the anchor month, feed that prediction in
as the next month's `TWS_t`, and carry on to the end of the run. `predict_recursive`
does this. Each step also picks up that month's *observed* SPEI and soil moisture — the
forcing between anchor and target is never masked, and nothing else in the pipeline uses
it. Simply carrying the anchor's `TWS_t` forward is far weaker (R² 0.17 at gap 1, and
negative by gap 3).

`mask_tws_like_test` reproduces the test layout so this is measurable, with runs never
straddling a calendar gap. Masking at random instead would leave same-month neighbours
visible and make neighbour-imputation look valuable when it can reach only 0.4% of real
masked rows.

Also fixed here: the median `SimpleImputer` is gone from `make_model()`, and the fit set
is augmented with a masked copy so the `NaN` branch is trained — a safety net for the
few cells the recursion cannot supply.


In [ ]:
## Lat/Lon not used as features
# Fit on original + randomly masked copies. Recursion supplies TWS_t almost everywhere,
# so this is now mostly a safety net for cells missing from the previous month.
fit_aug = augment_with_masking(fit_df, seed=0)

# Score against the real test layout: whole months masked, in runs.
val_masked = add_cell_id(mask_tws_like_test(val_df))
val_is_masked = val_masked["TWS_t_masked"].to_numpy()
val_gap = val_masked["gap"].to_numpy()

X_fit, y_fit, tws_fit = build_xy(fit_aug, feature_names, use_latlon=False)
X_val, y_val, tws_val = build_xy(val_masked, feature_names, use_latlon=False)

# Kept only to show what the unmasked split used to report.
X_val_full, _, tws_val_full = build_xy(val_df, feature_names, use_latlon=False)

print("fit rows (augmented):", X_fit.shape)
print("val rows:", X_val.shape, "| masked share:", f"{val_is_masked.mean():.1%}")
print("val runs (month counts):", [len(r) for r in find_runs(val_masked)])


In [ ]:
model = make_model()
model.fit(X_fit, y_fit)

# The three strategies, on identical rows.
y_recursive = predict_recursive(model, val_masked, feature_names)  # chain each run
y_nan = model.predict(X_val)                                       # feed NaN when masked
y_oracle = model.predict(X_val_full)                               # true TWS_t (unreachable)

# Persistence needs TWS_t; where masked, fall back to climatology (0 = mean anomaly).
y_persist = np.where(np.isnan(tws_val), 0.0, tws_val)

print("Recursive   <- the number to trust")
print(compute_metrics(y_val, y_recursive))
print("\nNaN when masked (what this notebook used to do)")
print(compute_metrics(y_val, y_nan))
print("\nOracle - true TWS_t everywhere (ceiling, not achievable)")
print(compute_metrics(y_val, y_oracle))
print("\nPersistence - climatology fallback where masked")
print(compute_metrics(y_val, y_persist))

print("\nR2 by forecast horizon")
print(f"  {'gap':>3s} {'n':>9s} {'recursive':>10s} {'NaN':>10s} {'oracle':>10s}")
for g in np.unique(val_gap):
    s = val_gap == g
    print(f"  {g:>3d} {s.sum():>9,} {r2_score(y_val[s], y_recursive[s]):>10.4f} "
          f"{r2_score(y_val[s], y_nan[s]):>10.4f} {r2_score(y_val[s], y_oracle[s]):>10.4f}")

# Recursion should shed amplitude gradually as confidence drops, not collapse.
print("\nPrediction spread by horizon")
print(f"  {'gap':>3s} {'std(true)':>10s} {'std(recur)':>11s} {'std(NaN)':>10s}")
for g in np.unique(val_gap):
    s = val_gap == g
    print(f"  {g:>3d} {y_val[s].std():>10.4f} {y_recursive[s].std():>11.4f} "
          f"{y_nan[s].std():>10.4f}")

y_val_pred = y_recursive  # downstream plots use the strategy we actually submit


In [ ]:
plt.figure(figsize=(6, 6))
n_plot = min(50000, len(y_val))
idx = np.random.RandomState(42).choice(len(y_val), size=n_plot, replace=False)

plt.scatter(y_val[idx], y_val_pred[idx], s=3, alpha=0.3)
mn = np.nanmin([y_val[idx].min(), y_val_pred[idx].min()])
mx = np.nanmax([y_val[idx].max(), y_val_pred[idx].max()])
plt.plot([mn, mx], [mn, mx], "--")
plt.xlabel("True TWS")
plt.ylabel("Predicted TWS")
plt.title("Validation predictions")
plt.grid(True)
plt.show()


In [ ]:
unique_val_times = np.sort(val_df["time"].unique())
time_example = unique_val_times[min(7, len(unique_val_times) - 1)]

plot_maps_from_predictions(
    val_df,
    y_val,
    y_val_pred,
    time_example
)


## 🚀 Fit the final model on all training data

Now refit on the full training set before predicting on the real test set.


In [ ]:
# Same masking augmentation as validation, so the submitted model has seen NaN TWS_t.
train_aug = augment_with_masking(train_features, seed=0)

X_train_full, y_train_full, tws_train_full = build_xy(
    train_aug,
    feature_names,
    use_latlon=False
)

print(X_train_full.shape, y_train_full.shape)

final_model = make_model()
final_model.fit(X_train_full, y_train_full)


## 📤 Build the submission

The flat test file already **is** the correct sample index (one row per official test
sample), so we align it to `SampleSubmission.csv` by ID and predict directly.

Predictions are generated **recursively**, run by run: the anchor month uses its observed
`TWS_t`, and each later month is fed the previous month's prediction — which is exactly
what its `TWS_t` would have been. `find_runs` recovers the run layout from the masking
pattern in the file itself, so nothing here is hard-coded to a particular release.


In [ ]:
sample_submission_df.head()


In [ ]:
test_features.head()

In [ ]:
### Really not necessary!
test_submission_df = sample_submission_df.merge(
    test_features,
    on="ID",
    how="left",
    validate="one_to_one"
)

print(test_submission_df.shape)
print(test_submission_df.isna().sum())
test_submission_df.head()


In [ ]:
test_pred_df = add_cell_id(test_submission_df)

print("test runs (month counts):", [len(r) for r in find_runs(test_pred_df)])

y_test_pred = predict_recursive(final_model, test_pred_df, feature_names)
assert not np.isnan(y_test_pred).any(), "some test rows were never predicted"

submission_df = sample_submission_df.copy()
submission_df["Target"] = y_test_pred.astype(np.float32)
submission_df.columns = ["ID", "Target"]

submission_df.head()


In [ ]:
SUBMISSION_PATH = DATA_DIR / "Benchmark.csv"
submission_df.to_csv(SUBMISSION_PATH, index=False)
print("Saved:", SUBMISSION_PATH)
